In [8]:
from transformers import MBartForConditionalGeneration , AutoTokenizer,AutoModelForSeq2SeqLM
import inspect
from transformers.models.mbart import modeling_mbart


In [2]:
model = AutoModelForSeq2SeqLM.from_pretrained("MarkSpaghetti/mbart-lora-r32-alpha-16")


Loading weights:   0%|          | 0/518 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [6]:
model

MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): MBartScaledWordEmbedding(250055, 1024, padding_idx=1)
    (encoder): MBartEncoder(
      (embed_tokens): MBartScaledWordEmbedding(250055, 1024, padding_idx=1)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x MBartEncoderLayer(
          (self_attn): MBartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [5]:
# Check dropout values set in the model config
config = model.config
dropout_attrs = {k: v for k, v in vars(config).items() if 'dropout' in k.lower()}
for attr, value in dropout_attrs.items():
    print(f"{attr}: {value}")

dropout: 0.1
attention_dropout: 0.0
activation_dropout: 0.0
classifier_dropout: 0.0
classif_dropout: 0.0


In [9]:
print(inspect.getsource(modeling_mbart.MBartEncoderLayer.forward))

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.Tensor,
        output_attentions: bool = False,
    ) -> torch.Tensor:
        """
        Args:
            hidden_states (`torch.FloatTensor`): input to the layer of shape `(batch, seq_len, embed_dim)`
            attention_mask (`torch.FloatTensor`): attention mask of size
                `(batch, 1, tgt_len, src_len)` where padding elements are indicated by very large negative values.
            output_attentions (`bool`, *optional*):
                Whether or not to return the attentions tensors of all attention layers. See `attentions` under
                returned tensors for more detail.
        """
        residual = hidden_states
        hidden_states = self.self_attn_layer_norm(hidden_states)
        hidden_states, attn_weights = self.self_attn(
            hidden_states=hidden_states,
            attention_mask=attention_mask,
            output_attentions=output_attentions

In [10]:

drop = 0.4574854
model.config.dropout = drop

# Or patch the attribute on all encoder/decoder layers directly:
for layer in model.model.encoder.layers:
    layer.dropout = drop

for layer in model.model.decoder.layers:
    layer.dropout = drop

In [11]:
# Check the actual p values on the first encoder layer
layer = model.model.encoder.layers[0]
print(f"dropout:             {layer.dropout}")
print(f"activation_dropout:  {layer.activation_dropout}")
print(f"model.training:      {model.training}")

dropout:             0.4574854
activation_dropout:  0.0
model.training:      False
